# 🏗️ CEM4644 · MP4 — Segmentation for quantity take-off on floor plans
## Homework (individual): *Residential floor plans, set B*

**No coding needed.** Each grey box below is one *step*: click the ▶ (play) button at its left, wait until it finishes, look at the result, then answer the report question that follows. Run the steps **from top to bottom**.

**What you will do (about 120 minutes)**
1. Meet SAM 3 on an ordinary site photo: ask by name, draw a box, tap an object. Then look at real floor plans and what is drawn on them.
2. Ask a segmentation model, by name, for rooms, fixtures and openings: see the mask, the overlay, the count and the area, and compare with the drawing's own numbers.
3. Do a quantity take-off with boxes, plan by plan: read the scale, count the windows, measure every room in square metres.
4. Examine where the model goes wrong: wording, weak regions, and your own words.
5. Try a plan of your own.

**Before you start:** menu *Runtime → Change runtime type → T4 GPU → Save*. The model used here (SAM 3) is large: with a GPU each request takes well under a second; without one, the precomputed results still work but live requests take about a minute each.

In [ ]:
#@title ▶ Step 0 · Run me first (2–3 minutes) { display-mode: "form" }
#@markdown Click ▶ and wait for the green ✅ line. This downloads the plans with their precomputed results and loads SAM 3 (about 3 GB).
#@markdown Untick *load_model* only if you have no GPU and want to skip the live steps.
load_model = True #@param {type:"boolean"}
import importlib, os, shutil, subprocess, sys
REPO, FOLDER, PKG = "CEM4644", "mp4_segmentation", "aec_seg"

def _git(*args):
    return subprocess.run(["git", "-C", REPO, *args], capture_output=True, text=True).returncode == 0

if os.path.isdir(REPO):                      # a copy is already here: pull the newest course code over it
    if not (_git("fetch", "-q", "--depth", "1", "origin", "master")
            and _git("reset", "-q", "--hard", "FETCH_HEAD") and _git("clean", "-qfd")):
        shutil.rmtree(REPO, ignore_errors=True)          # broken copy: start again from scratch
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "https://github.com/Haolan-Zhang/CEM4644.git", REPO], check=True)
for _m in [m for m in list(sys.modules) if m == PKG or m.startswith(PKG + ".")]:
    del sys.modules[_m]                      # Python caches imported code: drop it, or this cell keeps the old version
importlib.invalidate_caches()
sys.path.insert(0, os.path.abspath(os.path.join(REPO, FOLDER)))
from aec_seg import lab
lab.setup(dataset="homes_b", load_model=load_model)


## Part 1 · Meet SAM 3

Detection (MP3) draws a **box** around an object. **Segmentation** goes one step further: it decides, *pixel by pixel*, what belongs to the object. Count the pixels and you have an area; know the scale and you have square metres. That is what makes it useful for **quantity take-off** later in this notebook.

The model is **SAM 3** (Segment Anything Model 3, Meta 2025). You do not train it, and it has no fixed list of classes. You tell it *what* or *where*, in one of three ways:

- a **phrase**, such as *helmet* or *wet concrete*: it returns every region that matches, each with a **confidence**;
- a **box** around one object: it cuts out that object's exact outline;
- a **click** on one object: same thing, from a single point.

Two steps on an ordinary site photo first, so you see what the model does before it meets a drawing.

In [ ]:
#@title ▶ Step 1a · Ask by name { display-mode: "form" }
#@markdown Pick a phrase, or type your own in *own_phrase* (it wins when it is not empty). Three panels: the photo, the mask (white = the model says *this is it*), the overlay. Try a thing (*helmet*), a material (*wet concrete*), a part (*hand*), and something that is not there. Watch the confidences and move the slider.
photo = "pour: Workers pouring and levelling concrete on a slab" #@param ["pour: Workers pouring and levelling concrete on a slab", "mixer: A mixer truck delivering concrete next to a brick house"]
phrase = "person" #@param ["person", "helmet", "safety vest", "boots", "hose", "rebar", "wet concrete", "hand", "truck", "wheel", "brick wall", "sky", "window"]
own_phrase = "" #@param {type:"string"}
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.intro_phrase(photo, phrase, own_phrase, confidence)


In [ ]:
#@title ▶ Step 1b · Box it, or tap it { display-mode: "form" }
#@markdown Draw a box around an object and label it *box*; or draw a tiny box on an object and label it *point* (its centre is the click). Draw several, click *Submit*: SAM 3 cuts out one object per box or click, no words needed. Needs the live model.
photo = "pour: Workers pouring and levelling concrete on a slab" #@param ["pour: Workers pouring and levelling concrete on a slab", "mixer: A mixer truck delivering concrete next to a brick house"]
lab.intro_draw(photo)


### The plans

On a drawing the same three prompts work, and the answer key lets us check every result. Seven other real floor plans from CubiCasa5K (CC BY-NC-SA 4.0): larger houses, two-storey plans with both floors on one sheet, a Swedish-labelled plan and a small flat with printed room areas. Same scale bar, same answer keys.

Every plan has a **5 m scale bar** at the bottom left. Room labels are abbreviations in Finnish (one plan is Swedish):

| label | meaning |
|---|---|
| OH | olohuone = living room |
| MH | makuuhuone = bedroom |
| H | huone = room |
| K / KT / KEITTIÖ | keittiö = kitchen |
| KK | keittokomero = kitchenette |
| RT / RUOK | ruokailutila = dining area |
| KH / KPH | kylpyhuone = bathroom |
| PH / PESUH | pesuhuone = washroom |
| WC | toilet |
| S | sauna |
| ET | eteinen = entrance hall |
| TK | tuulikaappi = vestibule |
| KÄYTÄVÄ | corridor |
| VH | vaatehuone = walk-in closet |
| PUKUH | pukuhuone = dressing room |
| KHH | kodinhoitohuone = utility room |
| VAR / VARASTO | varasto = storage |
| TEKN | tekninen tila = technical room |
| PARVEKE / PARV | balcony |
| TERASSI | terrace |
| KUISTI | porch |
| ULKOTILA | outdoor area |
| AT / AUTOTALLI / AUTOKATOS | garage / carport |
| TUPA | farmhouse living room |
| SOVR / KÖK / BAD / HALL | Swedish: bedroom / kitchen / bathroom / hall |
| m² | square metres (printed on some plans) |

In [ ]:
#@title ▶ Step 1c · Browse the plans { display-mode: "form" }
#@markdown *all plans* shows every plan with what the drawing contains (rooms, floor area, doors, windows). These facts come from the plans' own annotations and are the answer key the notebook checks you against.
which = "all plans" #@param ["all plans", "14341: long single-storey house, 20 rooms", "5018: two-storey house: ground floor (left) and upper floor (right)", "1217: two-storey villa, Swedish labels", "8138: apartment with bold walls and furniture", "9136: two-storey house, both floors stacked on the sheet", "11615: small flat A3 with printed room areas", "10715: apartment (bold walls)"]
lab.show_plans(which)


## Part 2 · Ask for something by name

Pick a plan and a thing. You get three panels: the drawing, the **mask** (white = the model says *this is it*), and the **overlay**. Below them: how many regions, how many pixels, how many square metres (using the plan's scale), and what the **drawing's own answer key** says. The **confidence slider** hides the regions the model is unsure about: watch the count and the area change.

In [ ]:
#@title ▶ Step 2a · Original → mask → overlay { display-mode: "form" }
#@markdown Try *room (any)*, then *bedroom* and *bathroom*; then the symbols *toilet*, *sink*, *stairs*; then *kitchen*, *door* and *window*. Some words work, some find nothing at all: that is part of the lesson.
plan = "14341: long single-storey house, 20 rooms" #@param ["14341: long single-storey house, 20 rooms", "5018: two-storey house: ground floor (left) and upper floor (right)", "1217: two-storey villa, Swedish labels", "8138: apartment with bold walls and furniture", "9136: two-storey house, both floors stacked on the sheet", "11615: small flat A3 with printed room areas", "10715: apartment (bold walls)"]
thing = "bedroom" #@param ["room (any)", "bedroom", "bathroom", "kitchen", "living room", "balcony / terrace", "toilet", "sink", "bathtub", "stairs", "door", "window", "wall"]
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.segment(plan, thing, confidence)


In [ ]:
#@title ▶ Step 2b · Hits, misses and extras { display-mode: "form" }
#@markdown The answer key drawn on the plan: green = a real one the model found, red = a real one it missed, blue = a region that is not one.
plan = "14341: long single-storey house, 20 rooms" #@param ["14341: long single-storey house, 20 rooms", "5018: two-storey house: ground floor (left) and upper floor (right)", "1217: two-storey villa, Swedish labels", "8138: apartment with bold walls and furniture", "9136: two-storey house, both floors stacked on the sheet", "11615: small flat A3 with printed room areas", "10715: apartment (bold walls)"]
thing = "toilet" #@param ["room (any)", "bedroom", "bathroom", "kitchen", "living room", "balcony / terrace", "toilet", "sink", "bathtub", "stairs", "door", "window", "wall"]
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.count(plan, thing, confidence)


> ### 📝 Report question 1
> From Step 2a and 2b: which words found what they should (rooms? toilets? windows? doors?), and which found nothing or something else? Give the found / missed / extra counts for two things on one plan at confidence 0.3, and say what the misses have in common.

## Part 3 · Quantity take-off with boxes

A phrase is quick, but a take-off needs control. So now you draw the boxes, in one cell per plan: **the scale** (one box along the 5 m scale bar: a 1 % error in the scale is a 2 % error in every area), **every window** (a small box each), and **every room** (a tight box each, edges on the inside faces of the walls). Draw the small things first, so that later boxes do not overlap them. Then *Submit*: the notebook reads the scale from your box, counts your windows against the drawing's, measures every room with SAM 3 and prints each next to the drawing's own area.

How a *room* box is measured: a box alone makes SAM 3 cut out the *furniture symbols* inside it rather than the empty floor (it was trained to find objects). So the notebook asks for *empty room* **and** gives your box as the example, keeps the region that fits your box, fills the holes left by symbols, removes the black walls, and converts the pixels with **your** scale.

In [ ]:
#@title ▶ Step 3a · Scale, windows, rooms: the take-off of one plan { display-mode: "form" }
#@markdown Zoom with the mouse wheel. Order: *scale bar* (0 to 5 m), then every *window*, then every *room*, then *Submit*. Rooms need the live model. Do this for three plans of your choice, one of them a two-storey sheet and copy each table into your report.
plan = "14341: long single-storey house, 20 rooms" #@param ["14341: long single-storey house, 20 rooms", "5018: two-storey house: ground floor (left) and upper floor (right)", "1217: two-storey villa, Swedish labels", "8138: apartment with bold walls and furniture", "9136: two-storey house, both floors stacked on the sheet", "11615: small flat A3 with printed room areas", "10715: apartment (bold walls)"]
lab.takeoff(plan)


> ### 📝 Report question 2
> From Step 3a on three plans of your choice (one of them a two-storey sheet): your scale reading and its error, the table of rooms (your m², the drawing's m², the error), the total against the drawing's floor area, and the windows found / missed / extra. On plan 5018 the sheet prints KERROSALA and HUONEISTOALA (gross and net floor area per storey): how do they relate to what you measured?

## Part 4 · Where it goes wrong

Three kinds of error to look for: the **words** you use (the model was trained on everyday photos, not on drawings), **weak regions** it proposes with low confidence, and words of your own that describe what is *drawn* rather than what it *means*.

In [ ]:
#@title ▶ Step 4a · Does the wording matter? { display-mode: "form" }
#@markdown The same thing asked for with different words; all wordings are precomputed.
plan = "14341: long single-storey house, 20 rooms" #@param ["14341: long single-storey house, 20 rooms", "5018: two-storey house: ground floor (left) and upper floor (right)", "1217: two-storey villa, Swedish labels", "8138: apartment with bold walls and furniture", "9136: two-storey house, both floors stacked on the sheet", "11615: small flat A3 with printed room areas", "10715: apartment (bold walls)"]
thing = "stairs" #@param ["room (any)", "bedroom", "bathroom", "kitchen", "living room", "balcony / terrace", "toilet", "sink", "bathtub", "stairs", "door", "window", "wall"]
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.phrase_lab(plan, thing, confidence)


In [ ]:
#@title ▶ Step 4b · Look at each region and its confidence { display-mode: "form" }
#@markdown Every region the model proposed, numbered, with its confidence, its area and the room it sits on. Move the slider to see which ones survive.
plan = "14341: long single-storey house, 20 rooms" #@param ["14341: long single-storey house, 20 rooms", "5018: two-storey house: ground floor (left) and upper floor (right)", "1217: two-storey villa, Swedish labels", "8138: apartment with bold walls and furniture", "9136: two-storey house, both floors stacked on the sheet", "11615: small flat A3 with printed room areas", "10715: apartment (bold walls)"]
thing = "living room" #@param ["room (any)", "bedroom", "bathroom", "kitchen", "living room", "balcony / terrace", "toilet", "sink", "bathtub", "stairs", "door", "window", "wall"]
lab.inspect(plan, thing)


In [ ]:
#@title ▶ Step 4c · Your own words { display-mode: "form" }
#@markdown Type any phrase: a room, a symbol, a shape. Try *curved line* (the door swings), *thick black line* (the walls), *circle*, *small rectangle*. Needs the live model.
plan = "14341: long single-storey house, 20 rooms" #@param ["14341: long single-storey house, 20 rooms", "5018: two-storey house: ground floor (left) and upper floor (right)", "1217: two-storey villa, Swedish labels", "8138: apartment with bold walls and furniture", "9136: two-storey house, both floors stacked on the sheet", "11615: small flat A3 with printed room areas", "10715: apartment (bold walls)"]
phrase = "curved line" #@param {type:"string"}
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.your_phrase(plan, phrase, confidence)


> ### 📝 Report question 3
> From Step 4a: which wording worked best for the thing you chose, and how different were the counts and areas? From Step 4b: describe one weak region (what it sits on, its confidence) and one plain mistake, and what you would tell a colleague who wants to use these square metres in a cost estimate.

> ### 📝 Report question 4
> From Step 4c: which of your own words found something the named things could not (for example *curved line* for the door swings, *thick black line* for the walls)? Why does a shape word work on a drawing where the name of the thing does not?

## Part 5 · Your own plan

In [ ]:
#@title ▶ Your plan, your words { display-mode: "form" }
#@markdown This cell prints a **link**: open it in a new tab (or on your phone). Upload a floor plan (a photo of a drawing works too), type what to find, move the threshold. If the plan has a scale bar, measure how many pixels one metre is and enter the centimetres per pixel to get square metres.
#@markdown Test at least 3 plan(s) of your own and take screenshots for your report. Needs the live model.
lab.upload_app()


> ### 📝 Report question 5
> Test 3 plan(s) of your own (any floor plan from the internet or a course). For each: the phrase you used, the count and area measured, and whether the mask is right. What kind of drawing or wording failed?

> ### 📝 Report question 6
> Where in a project would a take-off like this be useful, and where would it mislead? What would you need (clean drawings, a scale, a room schedule, a person checking) to turn it into numbers you would put in an estimate?

## Wrap-up

In [ ]:
#@title ▶ Numbers for your report { display-mode: "form" }
lab.report_summary()


### Plan credits and model
- Site photos in Part 1: Workers pouring and levelling concrete on a slab (U.S. Air Force / Airman Sydney Franklin, Public domain, https://upload.wikimedia.org/wikipedia/commons/0/0e/Concrete_pouring_for_the_new_Spangdahlem_Elementary_School_%288062807%29.jpg); A mixer truck delivering concrete next to a brick house (Kolforn, CC BY-SA 4.0, https://upload.wikimedia.org/wikipedia/commons/5/57/-2021-01-18_Foundations_and_concrete_oversite%2C_Trimingham%2C_Norfolk_%283%29.JPG).
- Floor plans: CubiCasa5K (Kalervo, Ylioinas, Häikiö, Karhu, Kannala 2019), CubiCasa Oy, licence CC BY-NC-SA 4.0, https://zenodo.org/records/2613548. Sample ids in this notebook: 14341, 5018, 1217, 8138, 9136, 11615, 10715 (folder `data/plans/homes_b`, credits in `credits.json`). The plans were resampled to a plan-specific scale and given a scale bar; the answer keys come from the dataset's vector annotations.
- Model: SAM 3 by Meta AI (SAM License), loaded from a public mirror of the official checkpoint; a copy of the licence is in `docs/SAM_LICENSE.txt`.
- Lab code: https://github.com/Haolan-Zhang/CEM4644 (folder `mp4_segmentation`).